# SQL-каталог из Greenplum

Этот ноутбук самостоятельно читает таблицу Greenplum, предварительно агрегирует одинаковые пары исходного SQL и шаблона, строит снимок схемы каталога и запускает SQLGlot AST-анализ. SQL разбирается, но никогда не выполняется.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

ROOT = None
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "src" / "gp_sql_analyzer").is_dir():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Unable to locate project src/gp_sql_analyzer from the current working directory.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from gp_sql_analyzer.dataframe import analyze_dataframe
from gp_sql_analyzer.greenplum import (
    SourceQueryConfig,
    connect_greenplum,
    iter_greenplum_records,
    load_catalog_schema,
)


## 1. Конфигурация

`SOURCE_TABLE` должен содержать колонки `query_text` и `query_text_template`. Загрузчик оставляет только шаблоны с `&CHARACTER` и предварительно агрегирует одинаковые пары.

`BATCH_SIZE controls fetch size only`: он не ограничивает чтение на стороне сервера. Предагрегация всё ещё может сканировать и группировать весь подходящий источник, поэтому для пробных запусков рекомендуются фильтры. `ALLOW_FULL_SCAN=True` — явное согласие на полный скан. После фильтрации `query_records and DataFrames are materialized eagerly`, поэтому учитывайте память клиента.

In [ ]:
SOURCE_TABLE = "analytics.query_log"
DEFAULT_SCHEMA = "public"
BATCH_SIZE = 500
LIMIT = None
ID_COLUMN = None
SINCE_COLUMN = None
SINCE_VALUE = None
MIN_ID = None
MAX_ID = None
ALLOW_FULL_SCAN = False
OUTPUT_DIR = ROOT / "reports" / "greenplum"
BUILD_HTML = False


## 2. Только чтение из Greenplum

Подключение использует `GP_DSN` либо параметры окружения `GP_HOST`, `GP_PORT`, `GP_DBNAME`, `GP_USER`, `GP_PASSWORD`, `GP_SSLMODE`. Пароль в ноутбуке не задаётся. Сессия и запросы предназначены только для чтения.

In [ ]:
if type(ALLOW_FULL_SCAN) is not bool:
    raise TypeError("ALLOW_FULL_SCAN must be a boolean.")

has_complete_since_filter = SINCE_COLUMN is not None and SINCE_VALUE is not None
has_id_bound = ID_COLUMN is not None and (MIN_ID is not None or MAX_ID is not None)
if ALLOW_FULL_SCAN is not True and not (has_complete_since_filter or has_id_bound):
    raise RuntimeError(
        "An unbounded Greenplum scan is refused. Set SINCE_COLUMN/SINCE_VALUE, "
        "ID_COLUMN with MIN_ID/MAX_ID, or explicitly ALLOW_FULL_SCAN=True."
    )

source_config = SourceQueryConfig(
    table=SOURCE_TABLE,
    id_column=ID_COLUMN,
    since_column=SINCE_COLUMN,
    since_value=SINCE_VALUE,
    min_id=MIN_ID,
    max_id=MAX_ID,
    limit=LIMIT,
    preaggregate=True,
)
connection = connect_greenplum()
try:
    query_records = [record for batch in iter_greenplum_records(connection, source_config, batch_size=BATCH_SIZE) for record in batch]
    schema_provider = load_catalog_schema(connection, default_schema=DEFAULT_SCHEMA)
finally:
    connection.close()

queries_df = pd.DataFrame(
    [
        {
            "query_id": record.query_id,
            "query_text": record.query_text,
            "query_text_template": record.query_text_template,
            "source_row_count": record.source_row_count,
        }
        for record in query_records
    ],
    columns=["query_id", "query_text", "query_text_template", "source_row_count"],
)
schema_df = pd.DataFrame(
    [
        {
            "table_catalog": table.catalog,
            "table_schema": table.schema,
            "table_name": table.table,
            "column_name": column_name,
        }
        for table in schema_provider.tables
        for column_name in sorted(table.columns or ())
    ],
    columns=["table_catalog", "table_schema", "table_name", "column_name"],
).sort_values(
    ["table_catalog", "table_schema", "table_name", "column_name"],
    na_position="first",
).reset_index(drop=True)

if queries_df.empty:
    raise RuntimeError("Greenplum returned no query/template pairs containing &CHARACTER.")

print(f"Unique query/template pairs: {len(queries_df)}")
print(f"Weighted original source-row total: {queries_df['source_row_count'].sum()}")
print(f"Schema column count: {len(schema_df)}")


## 3. SQLGlot AST-анализ

`analyze_dataframe` разбирает оба текста SQL (`query_text` и `query_text_template`) в SQLGlot AST и не выполняет их.

In [ ]:
result = analyze_dataframe(
    queries_df,
    schema_df=schema_df,
    default_schema=DEFAULT_SCHEMA,
    output_dir=OUTPUT_DIR,
    build_html=BUILD_HTML,
    example_limit=20,
    source_label=SOURCE_TABLE,
)
row_analysis_df = result.row_analysis_df
details_df = result.details_df
aggregate_df = result.aggregate_df
catalog_tables_df = result.catalog_tables_df
catalog_columns_df = result.catalog_columns_df
errors_df = result.errors_df
assert len(row_analysis_df) == len(queries_df)

print(f"Parsed pair count: {len(row_analysis_df)}")
print(f"Details count: {len(details_df)}")
print(f"Aggregate count: {len(aggregate_df)}")


## 4. Результаты

Ниже приведены первые строки каждого результирующего DataFrame.

In [ ]:
display(row_analysis_df.head(20))
display(details_df.head(50))
display(aggregate_df.head(50))
display(catalog_tables_df.head(20))
display(catalog_columns_df.head(50))
display(errors_df.head(20))


## 5. Артефакты

HTML создаётся только когда `BUILD_HTML=True` и задан `OUTPUT_DIR`. `OUTPUT_DIR writes/replaces JSON artifacts`, даже когда HTML выключен; установите `OUTPUT_DIR=None` для запуска только в памяти.

In [ ]:
for name, path in result.artifact_paths.items():
    print(f"{name}: {path}")
